# Day 13 — 전처리와 QC: raw count, 정규화, log 변환은 각각 어디에 쓰나

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-13.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-13.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 한 유전자를 세 가지 값으로 보기

샘플 6개의 가상 count 표를 만듭니다. 모든 샘플의 **진짜 발현 수준은 같고**, 읽은 깊이(depth)만 0.5배에서 3배까지 다르게 했습니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(13)
n_genes = 500
samples = ["S1", "S2", "S3", "S4", "S5", "S6"]
depth = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 3.0])       # 샘플마다 '읽은 양'이 다르다

true_level = rng.lognormal(mean=3, sigma=1.5, size=n_genes)   # 모든 샘플에서 같은 진짜 발현 수준
mu = true_level[:, None] * depth[None, :]
counts = rng.negative_binomial(n=50, p=50 / (50 + mu))        # RNA 계수 데이터처럼 흩어진 정수
counts = pd.DataFrame(counts, index=[f"gene{i:03d}" for i in range(n_genes)], columns=samples)

lib_size = counts.sum()                          # 샘플별 총 count (library size)
cpm = counts / lib_size * 1_000_000              # counts per million
log_cpm = np.log2(cpm + 1)                       # +1은 log(0)을 피하기 위한 pseudocount

print("샘플별 총 count:")
print(lib_size)
print("\n한 유전자(gene000)를 세 가지 값으로 보기:")
print(pd.DataFrame({"raw": counts.loc["gene000"],
                    "CPM": cpm.loc["gene000"].round(0),
                    "log2(CPM+1)": log_cpm.loc["gene000"].round(2)}))

### 예시 2 — log 변환 전후, 누가 분산을 차지하나

예시 1을 실행한 같은 노트북에서 이어서 실행합니다.

In [ ]:
def top10_share(table):
    v = table.var(axis=1)                        # 유전자별 분산
    return v.sort_values(ascending=False).head(10).sum() / v.sum()

print(f"CPM에서 상위 10개 유전자가 차지하는 분산: {top10_share(cpm):.0%}")
print(f"log2(CPM+1)에서 상위 10개 유전자가 차지하는 분산: {top10_share(log_cpm):.0%}")

### 예시 3 — 샘플 QC 표 만들기

역시 같은 노트북에서 이어서 실행합니다.

In [ ]:
qc = pd.DataFrame({
    "total_count": counts.sum(),
    "detected_genes": (counts > 0).sum(),                 # count가 1 이상인 유전자 수
    "median_corr": log_cpm.corr(method="spearman").median(),   # 다른 샘플들과의 상관 중앙값
})
print(qc.round(3))

## 직접 해보기

1. 새 노트북 `notebooks/02_osdr_qc_and_pca.ipynb`를 만들고(3주차까지 계속 쓸 노트북입니다) 첫 섹션 제목을 `## Day 13 — 전처리와 QC 연습 (가상 데이터)`로 답니다. 예시 1–3을 차례로 실행합니다.
2. 세 가지 값의 샘플별 분포를 나란히 그립니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, table) in zip(axes, [("raw count", counts), ("CPM", cpm), ("log2(CPM+1)", log_cpm)]):
    sns.boxplot(data=table, ax=ax, showfliers=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("../figures/day13_three_scales.png", dpi=150)
plt.show()

raw count 그림에서는 상자가 S1에서 S6으로 갈수록 계단처럼 올라갑니다(샘플별 중앙값: 10, 17, 21, 26, 30, 62). log2(CPM+1) 그림에서는 여섯 상자의 중앙값이 9.0–9.1로 거의 같아집니다. 세 그림 중 어느 것이 "샘플끼리 비교 가능한 상태"인지 한 문장으로 적습니다.
3. 문제 있는 샘플을 일부러 만들어 QC 표가 잡아내는지 봅니다. 예시 1 뒤에 다음 한 줄을 추가하고 `lib_size`, `cpm`, `log_cpm` 계산과 예시 3을 다시 실행합니다.

In [ ]:
counts["S3"] = rng.poisson(75, n_genes)      # S3를 유전자와 무관한 잡음으로 바꿈 (총 count는 원래와 비슷하게)